# AURA Motor Model — Research Notebooks

These notebooks support the motor-skills modeling pipeline:

- **Dataset**: one row per session (your final `motor_sessions.csv`)
- **Goal**: train a *hostable* model that outputs an `impairment_score` in **[0,1]**
  - **0** = better motor performance in this task
  - **1** = more assistance needed in this task
- **Leakage control**: splits are grouped by **userId** (no user appears in train and test)

> ⚠️ This is a functional interaction score for this game task, **not a medical diagnosis**.


## 02 — Train & Cross-Validate (GroupKFold)

Runs the training script and inspects outputs.

Artifacts written to:
- `../model_registry/motor/1.0.0/`

In [ ]:
import os, subprocess, sys
from pathlib import Path

DATASET = os.environ.get("AURA_DATASET", "../datasets/final/motor_sessions.csv")
OUTDIR  = os.environ.get("AURA_OUTDIR", "../model_registry/motor/1.0.0")
FOLDS   = int(os.environ.get("AURA_FOLDS", "5"))

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
print("DATASET:", DATASET)
print("OUTDIR :", OUTDIR)
print("FOLDS  :", FOLDS)


In [ ]:
cmd = [sys.executable, "../training/train_motor_model_v2.py",
       "--csv", DATASET,
       "--outdir", OUTDIR,
       "--folds", str(FOLDS)]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError("Training failed")


In [ ]:
import json, os
report_path = os.path.join(OUTDIR, "reports", "training_report.json")
with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("B1 overall:", report["modelB1_motor_only"]["overall"])
print("B2 overall:", report["modelB2_motor_plus_context"]["overall"])
print("Sanity checks:", report.get("sanity_checks", {}))


In [ ]:
import pandas as pd, os
scores_path = os.path.join(OUTDIR, "reports", "sessions_with_scores.csv")
df_scores = pd.read_csv(scores_path)
df_scores[["sessionId","userId","impairment_score_A","impairment_score_B2_pred"]].head(10)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
x = df_scores["impairment_score_A"].values
y = df_scores["impairment_score_B2_pred"].values
plt.scatter(x, y, s=10)
plt.xlabel("impairment_score_A (target)")
plt.ylabel("B2 prediction")
plt.title("B2 predicted vs target (all rows)")
plt.show()

print("MAE:", float(np.mean(np.abs(x-y))))


## Next

Proceed to **03_error_analysis.ipynb** or **04_export_and_scoring_demo.ipynb**.
